# NuNER Dataset Preprocessing

Load and preprocess the [NuNER](https://huggingface.co/datasets/numind/NuNER) dataset for information extraction evaluation.

In [11]:
import ast
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset, DatasetDict

## Load Dataset

In [2]:
ds = load_dataset("numind/NuNER", split="full")
print(f"Loaded {len(ds):,} rows")
print(f"Columns: {ds.column_names}")

Loaded 1,000,000 rows
Columns: ['input', 'output']


## Basic EDA

In [3]:
df = ds.to_pandas()
print(f"Shape: {df.shape}")
print(f"\nNull counts:\n{df.isnull().sum()}")
print(f"\nDuplicates: {df.duplicated().sum():,}")
df.head()

Shape: (1000000, 2)

Null counts:
input     3
output    0
dtype: int64

Duplicates: 46


,input,output
0,"State University of New York Press, 1997.",['State University of New York Press <> Publis...
1,"A message from Katarzyna… for September 1, 2014.",['Katarzyna <> Person <> a specific individual...
2,Welcome to all you folks in the Washington DC ...,['Washington DC <> City <> Capital city of the...
3,A sharing session on 10 years of World Clean-U...,['World Clean-Up Day <> Event <> Annual global...
4,Want to know how to sharpen kitchen shears?,['kitchen shears <> kitchen utensil <> a cutti...


## Parse Output Column

Convert `"['entity <> type', ...]"` strings into structured lists of `{"entity": ..., "type": ...}` dicts.

In [4]:
def parse_entities(output_str: str) -> list[dict]:
    """Parse an output string into a list of {entity, type} dicts."""
    try:
        items = ast.literal_eval(output_str)
    except (ValueError, SyntaxError):
        return []
    entities = []
    for item in items:
        parts = item.split(" <> ", maxsplit=2)
        if len(parts) == 3:
            entities.append({"entity": parts[0].strip(), "type": parts[1].strip(), "description": parts[2].strip()})
    return entities


# Test on a few rows
for i in range(3):
    print(f"Input:  {df.iloc[i]['input'][:100]}")
    print(f"Parsed: {parse_entities(df.iloc[i]['output'])}")
    print()

Input:  State University of New York Press, 1997.
Parsed: [{'entity': 'State University of New York Press', 'type': 'Publisher', 'description': 'A publishing company affiliated with the State University of New York.'}]

Input:  A message from Katarzyna… for September 1, 2014.
Parsed: [{'entity': 'Katarzyna', 'type': 'Person', 'description': 'a specific individual'}, {'entity': 'September 1, 2014', 'type': 'Date', 'description': 'a specific date'}]

Input:  Welcome to all you folks in the Washington DC and Richmond area who heard my husband, comedian Chris
Parsed: [{'entity': 'Washington DC', 'type': 'City', 'description': 'Capital city of the United States'}, {'entity': 'Richmond', 'type': 'City', 'description': 'Capital of the Commonwealth of Virginia'}, {'entity': 'Christian Finnegan', 'type': 'Person', 'description': 'Comedian and husband of the speaker'}]



## Entity Type Analysis

In [5]:
type_counter = Counter()
parse_failures = 0

for output_str in df["output"]:
    parsed = parse_entities(output_str)
    if not parsed and output_str != "[]":
        parse_failures += 1
    for ent in parsed:
        type_counter[ent["type"]] += 1

print(f"Unique entity types: {len(type_counter)}")
print(f"Parse failures: {parse_failures:,}")
print(f"Total entities: {sum(type_counter.values()):,}")
print(f"\nTop 20 types:")
for etype, count in type_counter.most_common(50):
    print(f"  {etype:30s} {count:>10,}")

Unique entity types: 272301
Parse failures: 4,321
Total entities: 4,382,861

Top 20 types:
  Person                            153,000
  Location                          139,697
  Organization                       60,003
  Event                              56,364
  Product                            54,123
  Company                            49,003
  location                           47,246
  person                             41,639
  Action                             37,451
  Date                               36,876
  Time                               36,808
  action                             33,080
  product                            29,857
  Activity                           29,461
  organization                       25,549
  Object                             24,964
  Service                            23,827
  Technology                         22,812
  Country                            21,918
  activity                           21,066
  event                      

In [6]:
for etype, count in type_counter.most_common(200):
    print(f"  {etype:30s} {count:>10,}")

  Person                            153,000
  Location                          139,697
  Organization                       60,003
  Event                              56,364
  Product                            54,123
  Company                            49,003
  location                           47,246
  person                             41,639
  Action                             37,451
  Date                               36,876
  Time                               36,808
  action                             33,080
  product                            29,857
  Activity                           29,461
  organization                       25,549
  Object                             24,964
  Service                            23,827
  Technology                         22,812
  Country                            21,918
  activity                           21,066
  event                              20,850
  Place                              18,305
  time period                   

## Load Entity Split

Load the `entity` split so we can extract matching rows from both splits.

In [7]:
ds_entity = load_dataset("numind/NuNER", split="entity")
print(f"Entity split: {len(ds_entity):,} rows")
print(f"Columns: {ds_entity.column_names}")

# Sanity check: both splits should have the same length
assert len(ds) == len(ds_entity), "Split lengths differ!"

Entity split: 1,000,000 rows
Columns: ['input', 'output']


## Create Training & Test Splits

Use the **entity** split for all outputs.
- `domain_train`: 5,000 rows with technology entities (filtered via string match on full split)
- `test`: 400 rows with technology entities (non-overlapping with domain_train)
- `generic_train`: 5,000 rows randomly sampled from all rows (no filtering)

In [13]:
SEED = 42
DOMAIN_TRAIN_SIZE = 5000
GENERIC_TRAIN_SIZE = 5000
TEST_SIZE = 400

# Fast vectorized filter for tech rows (no ast.literal_eval needed)
tech_mask = df["output"].str.contains(" <> technology <> ", case=False, regex=False)
tech_indices = df.index[tech_mask].tolist()
print(f"Tech rows: {len(tech_indices):,}")

# Sample domain_train + test from tech rows (non-overlapping)
random.seed(SEED)
tech_sampled = random.sample(tech_indices, DOMAIN_TRAIN_SIZE + TEST_SIZE)
domain_train_indices = sorted(tech_sampled[:DOMAIN_TRAIN_SIZE])
test_indices = sorted(tech_sampled[DOMAIN_TRAIN_SIZE:])

# Generic: random sample from ALL rows (no filtering needed)
random.seed(SEED)
generic_train_indices = sorted(random.sample(range(len(ds_entity)), GENERIC_TRAIN_SIZE))

# Select from entity split
domain_train_ds = ds_entity.select(domain_train_indices)
generic_train_ds = ds_entity.select(generic_train_indices)
test_ds = ds_entity.select(test_indices)

print(f"\ndomain_train : {len(domain_train_ds):,} rows")
print(f"generic_train: {len(generic_train_ds):,} rows")
print(f"test         : {len(test_ds):,} rows")

# Spot-check
print("\n--- domain_train samples ---")
for i in range(2):
    print(f"  input:  {domain_train_ds[i]['input'][:80]}...")
    print(f"  output: {domain_train_ds[i]['output'][:80]}...")
    print()

print("--- generic_train samples ---")
for i in range(2):
    print(f"  input:  {generic_train_ds[i]['input'][:80]}...")
    print(f"  output: {generic_train_ds[i]['output'][:80]}...")
    print()

print("--- test samples ---")
for i in range(2):
    print(f"  input:  {test_ds[i]['input'][:80]}...")
    print(f"  output: {test_ds[i]['output'][:80]}...")
    print()

Tech rows: 28,042

domain_train : 5,000 rows
generic_train: 5,000 rows
test         : 400 rows

--- domain_train samples ---
  input:  PUTRAJAYA - Government information managed and stored in digital format must be ...
  output: ['Putrajaya <> Location', 'Government <> Organization', 'Digital format <> Techn...

  input:  They will often change you to another server if you complain....
  output: ['server <> technology', 'complain <> action']...

--- generic_train samples ---
  input:  Find West Texas A&M University reviews, tuition costs and how many students are ...
  output: ['West Texas A&M University <> University', 'reviews <> Review', 'tuition costs ...

  input:  The following year I was an Emerging Artist in resident at SAW, and we have kept...
  output: ['Emerging Artist <> Person/Artist', 'SAW <> Organization']...

--- test samples ---
  input:  Reason: Just looking for a server i can hang on when i have some free time, whil...
  output: ['server <> technology', 'community <>

## Save to Disk

In [15]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

for label, subset in [("domain_train", domain_train_ds),
                      ("generic_train", generic_train_ds),
                      ("test", test_ds)]:
    out_path = data_dir / f"{label}.parquet"
    subset.to_parquet(str(out_path))
    print(f"Saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB, {len(subset):,} rows)")

print("\nDone!")

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Saved data\domain_train.parquet (0.9 MB, 5,000 rows)


Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Saved data\generic_train.parquet (0.9 MB, 5,000 rows)


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved data\test.parquet (0.1 MB, 400 rows)

Done!
